# Day 2: Validation Rules and Data Quality Audit

This notebook shows how to define validation rules and create a simple audit report.

> Focus on detecting/reporting issues, not full cleaning implementation.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
repo_root = Path.cwd().resolve().parents[1]
df = pd.read_csv(repo_root / 'data' / 'raw' / 'week2_hr_employee_records_messy.csv')
print('Shape:', df.shape)
df.head()


## Step 1 - Define validation rules


In [ ]:
valid_departments={'D10','D20','D30','D40'}
valid_contract={'full-time','part-time','contractor','intern'}
valid_mgr={'M001','M002','M003','M004'}
rules={
'required_employee_id': df['employee_id'].isna() | (df['employee_id'].astype(str).str.strip()==''),
'age_between_0_120': (pd.to_numeric(df['age'], errors='coerce')<0) | (pd.to_numeric(df['age'], errors='coerce')>120) | pd.to_numeric(df['age'], errors='coerce').isna(),
'salary_non_negative': (pd.to_numeric(df['salary'], errors='coerce')<0) | pd.to_numeric(df['salary'], errors='coerce').isna(),
'hire_date_valid': pd.to_datetime(df['hire_date'], errors='coerce', format='mixed').isna(),
'duplicate_employee_id': df.duplicated(subset=['employee_id'], keep=False),
'known_department': ~df['department_id'].isin(valid_departments),
'contract_standardized': ~df['contract_type'].str.lower().isin(valid_contract),
'manager_reference_exists': df['manager_id'].isna() | ~df['manager_id'].isin(valid_mgr),
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T


## Step 2 - Row-level evidence


In [ ]:
for r,m in rules.items():
    print('\n', '='*70)
    print(r, 'affected rows:', int(m.sum()))
    display(df.loc[m].head(5))


## Step 3 - Build audit report


In [ ]:
sev={
'required_employee_id':'High','age_between_0_120':'High','salary_non_negative':'High',
'hire_date_valid':'Medium','duplicate_employee_id':'High','known_department':'Medium',
'contract_standardized':'Low','manager_reference_exists':'Medium'}
rec={
'required_employee_id':'Escalate and resolve missing IDs before joins.',
'age_between_0_120':'Verify with HR master records and correct source data.',
'salary_non_negative':'Investigate extract mapping and payroll source rules.',
'hire_date_valid':'Enforce standard date format at entry point.',
'duplicate_employee_id':'Investigate key-generation process and dedup policy for next chapter.',
'known_department':'Reconcile against department reference table.',
'contract_standardized':'Align labels to canonical categories.',
'manager_reference_exists':'Validate manager reference integrity and onboarding flow.'}
audit = pd.DataFrame([{
'issue_type':k,'affected_rows':int(v.sum()),'severity':sev[k],'recommended_next_action':rec[k]} for k,v in rules.items()])
audit


## Step 4 - Prioritized summary


In [ ]:
rank={'High':3,'Medium':2,'Low':1}
audit['rank']=audit['severity'].map(rank)
summary=audit.sort_values(['rank','affected_rows'], ascending=[False,False]).drop(columns=['rank'])
summary


In [ ]:
records_with_any = pd.concat(rules.values(), axis=1).any(axis=1).sum()
print(f'Total records: {len(df)}')
print(f'Records with >=1 issue: {records_with_any} ({records_with_any/len(df):.1%})')


## Function spotlight - reusable rule-audit pattern

In real projects, we keep rule logic in functions so audits are reproducible.


In [ ]:
def evaluate_rule(rule_name, mask, severity, recommendation):
    """Create one audit row from a validation mask.

    Parameters
    ----------
    rule_name : str
        Unique name of the rule.
    mask : pd.Series (bool)
        Boolean vector where True means rule violation.
    severity : str
        Suggested priority level (High, Medium, Low).
    recommendation : str
        Next action for data owner or engineering team.

    Returns
    -------
    dict
        A dictionary that can be turned into one row of an audit report.
    """
    return {
        'issue_type': rule_name,
        'affected_rows': int(mask.sum()),
        'severity': severity,
        'recommended_next_action': recommendation,
    }


def build_audit_report(rule_dict, severity_map, recommendation_map):
    """Build a full audit report table from rule definitions.

    Parameters
    ----------
    rule_dict : dict[str, pd.Series]
        Mapping from rule name to violation mask.
    severity_map : dict[str, str]
        Severity per rule.
    recommendation_map : dict[str, str]
        Recommendation text per rule.

    Returns
    -------
    pd.DataFrame
        Audit report with one row per rule.
    """
    rows = [
        evaluate_rule(name, mask, severity_map[name], recommendation_map[name])
        for name, mask in rule_dict.items()
    ]
    return pd.DataFrame(rows)


audit_v2 = build_audit_report(rules, sev, rec)
audit_v2


## Governance reflection

1. Which high-severity issue should be fixed first?
2. Which rules should run automatically in ingestion?
3. How does this audit support accountability?


## Recap

Validation rules create measurable controls, while audits translate technical findings into governance-ready decisions.
